# Hotel Booking Cancellation Prediction
## Sprint 4: Deployment
**Goal: Make the model usable in real-world applications**

This sprint uses the artifacts produced in Sprints 1–3:
- `models/preprocessor.pkl` — fitted `ColumnTransformer` (Sprint 1)
- `src/feature_engineering.py` — `FeatureEngineer` & `FeatureSelector` classes (Sprint 3)
- `models/final_model.pkl` — tuned Stacking Classifier (Sprint 3)
- `models/selected_features.json` — top-30 selected features (Sprint 3)
- `preprocessed_df.csv` — preprocessed dataset (Sprint 1)

# Setup and Imports

In [7]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import csv
import sys
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

sys.path.append('src')
from feature_engineering import FeatureEngineer, FeatureSelector

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

pd.set_option('display.max_columns', None)
print('Imports successful')

Imports successful


---
# Step 1: Build End-to-End ML Pipeline

We chain together the 4 Sprint 1–3 artifacts into a single `sklearn.Pipeline` that takes **raw booking data** and outputs a **cancellation prediction**:

```
raw data --> ColumnTransformer --> DataFrame --> FeatureEngineer --> FeatureSelector --> Final Model
```

In [8]:
preprocessor = joblib.load('models/preprocessor.pkl')
final_model  = joblib.load('models/final_model.pkl')

with open('models/selected_features.json') as f:
    selected_features = json.load(f)

print('Preprocessor loaded:', type(preprocessor).__name__)
print('Final model loaded: ', type(final_model).__name__)
print('Selected features:  ', len(selected_features))

Preprocessor loaded: ColumnTransformer
Final model loaded:  StackingClassifier
Selected features:   236


### 1.2 Recover `adr_upper` (ADR Outlier Cap) Without Needing the Raw CSV

The `ct` (ColumnTransformer) was fit on data where `adr` had already been **capped** at an IQR-based upper bound (`adr_upper`) and then **scaled** by a `StandardScaler`. We don't need the original raw CSV to recover `adr_upper` — we can invert the fitted `StandardScaler`:

```
scaled_value = (capped_value - mean) / std
=> capped_value = scaled_value * std + mean
```

The **maximum** value of `sc__adr_capped` in `preprocessed_df.csv` corresponds to `adr_upper` itself (since any `adr` above the cap was set exactly to `adr_upper` before scaling).

In [12]:
df_pp = pd.read_csv(r"C:\Users\laksh\ML\ML Project\preprocessed_df")
df_pp.drop('sc__days_in_waiting_list.1',axis=1,inplace=True)

# Locate the 'adr_capped' column inside the fitted StandardScaler ('sc' transformer)
sc_transformer = preprocessor.named_transformers_['sc']
sc_columns = list(sc_transformer.feature_names_in_)
adr_capped_idx = sc_columns.index('adr_capped')

mean_ = sc_transformer.mean_[adr_capped_idx]
std_  = sc_transformer.scale_[adr_capped_idx]

# Max scaled value -> corresponds to the capped upper bound
max_scaled_adr = df_pp['sc__adr_capped'].max()
adr_upper = max_scaled_adr * std_ + mean_

print(f'Recovered ADR cap (adr_upper): {adr_upper:.2f}')

# Save it for reuse (e.g. in the Streamlit app)
os.makedirs('models', exist_ok=True)
with open('models/adr_upper_bound.json', 'w') as f:
    json.dump({'adr_upper': float(adr_upper)}, f)

Recovered ADR cap (adr_upper): 226.87


### 1.3 Recreate Pre-Processing Transformations

**Why this is needed:** the saved `ct` (ColumnTransformer) was fit on a dataframe that already had two derived columns added (`lead_time_log = log1p(lead_time)` and `adr_capped`, an outlier-capped version of `adr`), plus the `is_canceled` target column (passed through as `remainder`). A truly raw CSV would have `lead_time` and `adr` instead, and no `is_canceled`. We add a **`RawPreprocessor`** step at the very start of the pipeline that recreates these columns from raw input, exactly as Sprint 1 did.

In [13]:
from sklearn.base import BaseEstimator, TransformerMixin

os.makedirs('src', exist_ok=True)

raw_preprocessor_code = '''

class RawPreprocessor(BaseEstimator, TransformerMixin):
    \"\"\"
    Replicates the Sprint 1 transformations applied to the RAW dataframe
    BEFORE the ColumnTransformer (ct) was fit:

      - lead_time_log = log1p(lead_time)
      - adr_capped    = adr capped at the IQR-based upper bound
                        learned from the training data (adr_upper)
      - adds a placeholder 'is_canceled' column if missing
        (ct passes this through as `remainder__is_canceled`,
         but it is dropped before reaching the model)
      - selects exactly the 19 columns ct was fitted on
    \"\"\"

    REQUIRED_COLUMNS = [
        'hotel', 'is_canceled', 'arrival_date_month',
        'stays_in_weekend_nights', 'stays_in_week_nights',
        'adults', 'children', 'country', 'market_segment',
        'is_repeated_guest', 'previous_cancellations',
        'previous_bookings_not_canceled', 'reserved_room_type',
        'assigned_room_type', 'deposit_type', 'days_in_waiting_list',
        'customer_type', 'lead_time_log', 'adr_capped'
    ]

    def __init__(self, adr_upper):
        self.adr_upper = adr_upper

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X['lead_time_log'] = np.log1p(X['lead_time'])
        X['adr_capped'] = np.where(X['adr'] > self.adr_upper, self.adr_upper, X['adr'])

        if 'is_canceled' not in X.columns:
            X['is_canceled'] = 0

        if 'children' in X.columns:
            X['children'] = X['children'].astype(float)

        return X[self.REQUIRED_COLUMNS]
'''

# Write/overwrite the RawPreprocessor in the feature_engineering module
import re
existing_code = ''
if os.path.exists('src/feature_engineering.py'):
    with open('src/feature_engineering.py') as f:
        existing_code = f.read()

# Remove any previously-appended RawPreprocessor definition to avoid duplicates
existing_code = re.sub(r'\n*class RawPreprocessor.*', '', existing_code, flags=re.DOTALL)

# Ensure numpy is imported in the module (RawPreprocessor uses np.log1p / np.where)
if 'import numpy as np' not in existing_code:
    existing_code = 'import numpy as np\n' + existing_code

with open('src/feature_engineering.py', 'w') as f:
    f.write(existing_code + raw_preprocessor_code)

print('RawPreprocessor written to src/feature_engineering.py')

RawPreprocessor written to src/feature_engineering.py


In [14]:
# Reload the module to pick up the (re)written RawPreprocessor class
import importlib, sys
sys.path.append('src')
import feature_engineering
importlib.reload(feature_engineering)
from feature_engineering import FeatureEngineer, FeatureSelector, RawPreprocessor

raw_preprocessor = RawPreprocessor(adr_upper=adr_upper)
print('RawPreprocessor ready with adr_upper =', round(adr_upper, 2))

RawPreprocessor ready with adr_upper = 226.87


### 1.4 Wrap the ColumnTransformer output back into a DataFrame
`ColumnTransformer.transform()` returns a numpy array. `FeatureEngineer` and `FeatureSelector` both expect a `pandas.DataFrame` with named columns (e.g. `sc__lead_time_log`), so we add a small `FunctionTransformer` step that rebuilds the DataFrame using `preprocessor.get_feature_names_out()`.

In [15]:
def dedupe_columns(columns):
    """
    Replicates pandas' CSV-reading deduplication behavior: the first
    occurrence of a column name keeps its name, subsequent duplicates
    get '.1', '.2', etc. appended.

    Needed because `ct`'s StandardScaler was fit with
    'days_in_waiting_list' listed twice, so get_feature_names_out()
    returns 'sc__days_in_waiting_list' twice. preprocessed_df.csv
    (read via pd.read_csv) had these deduped to
    'sc__days_in_waiting_list' and 'sc__days_in_waiting_list.1' -
    we replicate that here so column names match selected_features.
    """
    seen = {}
    new_cols = []
    for col in columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f'{col}.{seen[col]}')
    return new_cols


feature_names = dedupe_columns(list(preprocessor.get_feature_names_out()))

print('Total columns:', len(feature_names))
print('Duplicates remaining:', len(feature_names) != len(set(feature_names)))
print([c for c in feature_names if 'days_in_waiting_list' in c])

Total columns: 233
Duplicates remaining: False
['sc__days_in_waiting_list']


### 1.4b: Save `ToDataFrame` as a Reusable Class

To avoid the previous `NameError: preprocessor not defined` issue (caused by a function closure referencing a notebook-only variable), we define `ToDataFrame` as a class that stores the column names as an **instance attribute**. This makes it fully self-contained when pickled — `app.py` won't need to recreate any `preprocessor`/`to_dataframe` variables.

In [16]:
import re

to_dataframe_code = '''

class ToDataFrame(BaseEstimator, TransformerMixin):
    \"\"\"
    Converts the numpy/sparse output of a ColumnTransformer back into a
    pandas DataFrame with the given (already-deduplicated) column names.

    Stores `feature_names` as an instance attribute (not a closure), so
    this transformer is fully self-contained when pickled - no external
    globals (like a `preprocessor` variable) are needed at unpickle time.
    \"\"\"

    def __init__(self, feature_names):
        self.feature_names = list(feature_names)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if hasattr(X, 'toarray'):
            X = X.toarray()
        return pd.DataFrame(X, columns=self.feature_names)
'''

existing_code = ''
if os.path.exists('src/feature_engineering.py'):
    with open('src/feature_engineering.py') as f:
        existing_code = f.read()

# Remove any previous ToDataFrame definition to avoid duplicates
existing_code = re.sub(r'\n*class ToDataFrame.*?(?=\nclass |\Z)', '', existing_code, flags=re.DOTALL)

# Ensure pandas is imported
if 'import pandas as pd' not in existing_code:
    existing_code = 'import pandas as pd\n' + existing_code

with open('src/feature_engineering.py', 'w') as f:
    f.write(existing_code + to_dataframe_code)

print('ToDataFrame written to src/feature_engineering.py')

ToDataFrame written to src/feature_engineering.py


In [17]:
# Reload the module to pick up ToDataFrame.
# IMPORTANT: re-import AND re-instantiate everything from THIS reload,
# so all classes used inside full_pipeline come from the same module
# version (avoids 'PicklingError: not the same object as ...').
importlib.reload(feature_engineering)
from feature_engineering import FeatureEngineer, FeatureSelector, RawPreprocessor, ToDataFrame

raw_preprocessor = RawPreprocessor(adr_upper=adr_upper)
to_df_transformer = ToDataFrame(feature_names)

print('RawPreprocessor and ToDataFrame ready (single reload, consistent class identity)')

RawPreprocessor and ToDataFrame ready (single reload, consistent class identity)


### 1.5 Assemble the Full Pipeline

In [18]:
full_pipeline = Pipeline([
    ('raw_preprocessing',   raw_preprocessor),
    ('preprocessing',       preprocessor),
    ('to_dataframe',        to_df_transformer),
    ('feature_engineering', FeatureEngineer()),
    ('feature_selection',   FeatureSelector(selected_features)),
    ('model',               final_model)
])

print(full_pipeline)

Pipeline(steps=[('raw_preprocessing',
                 RawPreprocessor(adr_upper=np.float64(226.87499999999997))),
                ('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['hotel',
                                                   'arrival_date_month',
                                                   'children', 'country',
                                                   'market_segment',
                                                   'reserved_room_type',
                                                   'assigned_room_type',
                                                   'deposit_typ...
                                          'ohe__arrival_date_month_October',


### 1.6 Test the Pipeline on Sample Raw Data

We test `full_pipeline.predict()` with a hand-built raw booking record — this uses the **same raw column names** (`hotel`, `lead_time`, `adr`, etc.) that the Streamlit form in Step 2 will collect. This works regardless of whether you have the original raw CSV available.

In [19]:
# End-to-end sanity check using the SAME dict structure as app.py's input_dict
test_input = pd.DataFrame([{
    'hotel':                          'Resort Hotel',
    'lead_time':                      90,
    'arrival_date_month':             'July',
    'stays_in_weekend_nights':        1,
    'stays_in_week_nights':           2,
    'adults':                         2,
    'children':                       0,
    'country':                        'PRT',
    'market_segment':                 'Online TA',
    'reserved_room_type':             'A',
    'assigned_room_type':             'A',
    'deposit_type':                   'No Deposit',
    'customer_type':                  'Transient',
    'adr':                            100.0,
    'previous_cancellations':         0,
    'previous_bookings_not_canceled': 0,
    'days_in_waiting_list':           0,
    'is_repeated_guest':              0,
}])

pred  = full_pipeline.predict(test_input)[0]
proba = full_pipeline.predict_proba(test_input)[0][1]
print(f'Prediction: {"Canceled" if pred == 1 else "Not Canceled"}  |  Probability: {proba:.4f}')

Prediction: Canceled  |  Probability: 0.7016


In [14]:
print("Selected features:")
print([c for c in selected_features if 'days_in_waiting_list' in c])

Selected features:
['sc__days_in_waiting_list']


### 1.7 (Optional) Test on the Original Raw Dataset

If you have the **original raw Kaggle 'Hotel Booking Demand' CSV** (with columns like `hotel`, `lead_time`, `adr`, `is_canceled`, etc. — *not* the preprocessed `ohe__`/`sc__` file), place it in the working directory and update the filename below to run a larger validation, including accuracy/F1/ROC-AUC against the held-out test set. Otherwise, skip this section — the pipeline is already validated above and saved in the next step.

In [20]:
RAW_DATA_PATH = r"C:\Users\laksh\ML\ML Project\preprocessed_df"  # <-- update this to your raw file's name/path

raw_y = None
raw_X_test = None
y_pred_pipeline = None
y_prob_pipeline = None

required_raw_columns = [
    'hotel', 'lead_time', 'arrival_date_month', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adults', 'children', 'country', 'market_segment',
    'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled',
    'reserved_room_type', 'assigned_room_type', 'deposit_type',
    'days_in_waiting_list', 'customer_type', 'adr'
]

if os.path.exists(RAW_DATA_PATH):
    raw_df = pd.read_csv(RAW_DATA_PATH, sep=None, engine='python')
    missing = [c for c in required_raw_columns if c not in raw_df.columns]

    if missing:
        print(f'{RAW_DATA_PATH} found but missing columns: {missing}')
        print('Skipping raw-data validation.')
    else:
        raw_X = raw_df.drop(columns=['is_canceled']) if 'is_canceled' in raw_df.columns else raw_df.copy()
        raw_y = raw_df['is_canceled'] if 'is_canceled' in raw_df.columns else None

        sample_raw = raw_X.sample(10, random_state=1)
        print('Sample predictions:', full_pipeline.predict(sample_raw))

        if raw_y is not None:
            raw_X_train, raw_X_test, raw_y_train, raw_y_test = train_test_split(
                raw_X, raw_y, test_size=0.2, random_state=42, stratify=raw_y
            )
            y_pred_pipeline = full_pipeline.predict(raw_X_test)
            y_prob_pipeline = full_pipeline.predict_proba(raw_X_test)[:, 1]

            print('=== Full Pipeline Evaluation (raw data in) ===')
            print(f'Accuracy: {accuracy_score(raw_y_test, y_pred_pipeline):.4f}')
            print(f'F1 Score: {f1_score(raw_y_test, y_pred_pipeline):.4f}')
            print(f'ROC-AUC:  {roc_auc_score(raw_y_test, y_prob_pipeline):.4f}')
else:
    print(f'{RAW_DATA_PATH} not found - skipping optional raw-data validation.')
    print('The pipeline has already been validated with a sample record in Step 1.6.')

C:\Users\laksh\ML\ML Project\preprocessed_df found but missing columns: ['hotel', 'lead_time', 'arrival_date_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'country', 'market_segment', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'days_in_waiting_list', 'customer_type', 'adr']
Skipping raw-data validation.


### 1.8 Save the Full Pipeline

In [21]:
joblib.dump(full_pipeline, 'models/full_pipeline.pkl')
print('Full end-to-end pipeline saved: models/full_pipeline.pkl')

# Quick reload check using the same sample record from 1.6
reloaded = joblib.load('models/full_pipeline.pkl')
print('Reload check prediction:', reloaded.predict(test_input))

Full end-to-end pipeline saved: models/full_pipeline.pkl
Reload check prediction: [1.]


---
# Step 2: Frontend — Streamlit App

Build a UI that takes raw booking details from a user, runs them through `full_pipeline.pkl`, and displays the cancellation prediction.

In [22]:
os.makedirs('app', exist_ok=True)

streamlit_code = "import streamlit as st\nimport pandas as pd\nimport joblib\nimport sys\nimport os\n\n# app.py lives in <project_root>/app/, while src/ and models/ live in <project_root>/\n# Make '<project_root>/src' importable so joblib can find RawPreprocessor,\n# FeatureEngineer, FeatureSelector, and ToDataFrame when unpickling full_pipeline.pkl\nAPP_DIR = os.path.dirname(os.path.abspath(__file__))\nPROJECT_ROOT = os.path.dirname(APP_DIR)\nsys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))\n\n# Importing these makes the classes available under the 'feature_engineering'\n# module name, which is what full_pipeline.pkl was pickled with.\nfrom feature_engineering import FeatureEngineer, FeatureSelector, RawPreprocessor, ToDataFrame\n\nPIPELINE_PATH = os.path.join(PROJECT_ROOT, 'models', 'full_pipeline.pkl')\n\nst.set_page_config(page_title='Hotel Booking Cancellation Predictor', page_icon='H', layout='wide')\n\n@st.cache_resource\ndef load_pipeline():\n    return joblib.load(PIPELINE_PATH)\n\npipeline = load_pipeline()\n\nst.title('Hotel Booking Cancellation Predictor')\nst.markdown('Enter raw booking details below to predict whether the booking is likely to be **cancelled**.')\nst.divider()\n\nst.sidebar.header('Booking Details')\n\nhotel               = st.sidebar.selectbox('Hotel Type', ['Resort Hotel', 'City Hotel'])\nlead_time           = st.sidebar.slider('Lead Time (days)', 0, 737, 90)\narrival_month       = st.sidebar.selectbox('Arrival Month', [\n    'January','February','March','April','May','June',\n    'July','August','September','October','November','December'\n])\nstays_weekend       = st.sidebar.slider('Weekend Nights', 0, 20, 1)\nstays_week          = st.sidebar.slider('Weekday Nights', 0, 50, 2)\nadults              = st.sidebar.slider('Adults', 1, 10, 2)\nchildren            = st.sidebar.selectbox('Children', [0, 1, 2, 3])\ncountry             = st.sidebar.text_input('Country Code (e.g. PRT, GBR, USA)', 'PRT')\nmarket_segment      = st.sidebar.selectbox('Market Segment', [\n    'Direct','Corporate','Online TA','Offline TA/TO','Complementary','Groups','Aviation'\n])\nreserved_room_type  = st.sidebar.selectbox('Reserved Room Type', ['A','B','C','D','E','F','G','H','L'])\nassigned_room_type  = st.sidebar.selectbox('Assigned Room Type', ['A','B','C','D','E','F','G','H','I','K','L'])\ndeposit_type        = st.sidebar.selectbox('Deposit Type', ['No Deposit','Refundable','Non Refund'])\ncustomer_type       = st.sidebar.selectbox('Customer Type', ['Transient','Contract','Transient-Party','Group'])\nadr                 = st.sidebar.number_input('Average Daily Rate (ADR)', 0.0, 600.0, 100.0)\nprev_cancellations  = st.sidebar.slider('Previous Cancellations', 0, 26, 0)\nprev_not_cancelled  = st.sidebar.slider('Previous Bookings Not Cancelled', 0, 72, 0)\ndays_waiting        = st.sidebar.slider('Days in Waiting List', 0, 391, 0)\nis_repeated_guest   = st.sidebar.selectbox('Repeated Guest?', [0, 1], format_func=lambda x: 'Yes' if x else 'No')\n\n# Build a single-row raw DataFrame. full_pipeline.predict() handles ALL preprocessing\n# (lead_time_log, adr_capped, OHE/scaling, feature engineering, feature selection).\ninput_dict = {\n    'hotel':                          hotel,\n    'lead_time':                      lead_time,\n    'arrival_date_month':             arrival_month,\n    'stays_in_weekend_nights':        stays_weekend,\n    'stays_in_week_nights':           stays_week,\n    'adults':                         adults,\n    'children':                       children,\n    'country':                        country,\n    'market_segment':                 market_segment,\n    'reserved_room_type':             reserved_room_type,\n    'assigned_room_type':             assigned_room_type,\n    'deposit_type':                   deposit_type,\n    'customer_type':                  customer_type,\n    'adr':                            adr,\n    'previous_cancellations':         prev_cancellations,\n    'previous_bookings_not_canceled': prev_not_cancelled,\n    'days_in_waiting_list':           days_waiting,\n    'is_repeated_guest':              is_repeated_guest,\n}\n\ninput_df = pd.DataFrame([input_dict])\n\nif st.sidebar.button('Predict', type='primary'):\n    pred  = pipeline.predict(input_df)[0]\n    proba = pipeline.predict_proba(input_df)[0][1]\n\n    st.subheader('Prediction Result')\n    col1, col2, col3 = st.columns(3)\n    with col1:\n        if pred == 1:\n            st.error('Likely to CANCEL')\n        else:\n            st.success('Likely NOT to Cancel')\n    with col2:\n        st.metric('Cancellation Probability', f'{proba*100:.1f}%')\n    with col3:\n        st.metric('Confidence', 'High' if abs(proba - 0.5) > 0.3 else 'Medium')\n\n    st.divider()\n    st.subheader('Booking Summary')\n    st.dataframe(input_df.T.rename(columns={0: 'Value'}), use_container_width=True)\nelse:\n    st.info('Fill in the booking details on the left and click Predict.')\n"

with open('app/app.py', 'w') as f:
    f.write(streamlit_code)

print('Streamlit app written to app/app.py')
print('Run from project root with: streamlit run app/app.py')

Streamlit app written to app/app.py
Run from project root with: streamlit run app/app.py
